# Train YOLO-Pose on MTDDH (Colab)

Notebook version of `scripts/train_yolo.py`.
Run cells in order: **Runtime → Change runtime type → GPU** first.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/dl-project/

## 2. Install & check environment
Training only needs `ultralytics` (mmpose is only needed for evaluation). Keep Colab's preinstalled torch.

In [ ]:
!pip install -q ultralytics

import torch, ultralytics
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
ultralytics.checks()

## 3. Config
Edit these instead of command-line arguments.
- `DATA_ZIP`: dataset zip on your Drive
- `PROJECT`: output root on Drive, so runs survive a disconnect
- `RESUME`: set to a `last.pt` path to resume an interrupted run

In [ ]:
from pathlib import Path

# paths
DATA_ZIP = "./data/mtddh_xray_2d_course_60_20_20.zip"  # TODO: your zip
DATA_DIR = Path("/content/data")                       # unzip to local disk (much faster than Drive)
PROJECT  = "./runs"    # results saved to Drive
NAME     = "y11n_640"
RESUME   = None   # e.g. f"{PROJECT}/{NAME}/weights/last.pt"

# model / hyperparameters
MODEL     = "yolo11n-pose.pt"
DEVICE    = None      # None = auto (0 / mps / cpu)
EPOCHS    = 100
IMGSZ     = 640
BATCH     = 16
OPTIMIZER = "auto"    # SGD / Adam / AdamW / auto  (auto ignores LR0)
LR0       = 0.01
PATIENCE  = 100       # early-stopping patience in epochs
SEED      = 0
CACHE     = True      # cache images in RAM
FRACTION  = 1.0       # e.g. 0.1 for a quick smoke test

## 4. Prepare dataset
Copy the zip to local disk, unzip, then rewrite `path:` in `data.yaml` to an absolute path
(`path: .` would otherwise resolve against the working directory, not the yaml's folder).

In [ ]:
import shutil, zipfile, yaml

if not any(DATA_DIR.rglob("data.yaml")):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    local_zip = Path("/content") / Path(DATA_ZIP).name
    shutil.copy(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(DATA_DIR)

DATA_YAML = next(p for p in DATA_DIR.rglob("data.yaml") if "__MACOSX" not in p.parts)
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg["path"] = str(DATA_YAML.parent.resolve())
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

print("data.yaml:", DATA_YAML)
print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
for split in ("train", "val", "test"):
    n = len(list((DATA_YAML.parent / cfg[split]).glob("*")))
    print(f"{split:5s}: {n} images")

## 5. Train

In [ ]:
from ultralytics import YOLO


def pick_device(device):
    if device is not None:
        return device
    if torch.cuda.is_available():
        return 0
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


if RESUME:
    # all settings are restored from the checkpoint's saved args
    model = YOLO(RESUME)
    model.train(resume=True)
else:
    model = YOLO(MODEL if MODEL.endswith(".pt") else MODEL + ".pt")
    model.train(
        data=str(DATA_YAML),
        device=pick_device(DEVICE),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        optimizer=OPTIMIZER,
        lr0=LR0,
        patience=PATIENCE,
        seed=SEED,
        cache=CACHE,
        fraction=FRACTION,
        project=PROJECT,
        name=NAME,
    )

## 6. Outputs
Copy `best weights` into `evaluate.py`.

In [ ]:
print(f"best weights: {model.trainer.best}")
print(f"run dir:      {model.trainer.save_dir}")

## 7. Evaluation

In [ ]:
!pip install -q "mmengine>=0.4.0,<1.0.0" "mmcv-lite>=2.0.0,<3.0.0"
!pip install -q --no-deps "mmpose==1.3.2"

In [ ]:
import sys
SRC_DIR = "/content/drive/MyDrive/dl-project/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

In [ ]:
from dl_project.metrics import NME, PCK